# Spindle — Quick Start

**Spindle** generates statistically realistic, relationally correct synthetic data for Microsoft Fabric.

This notebook covers:
- Installing and importing Spindle
- Generating the Retail domain at `fabric_demo` scale
- Exploring the resulting DataFrames
- Verifying referential integrity
- Overriding distributions at runtime

In [ ]:
%pip install sqllocks-spindle --quiet

## Generate retail data

The `fabric_demo` scale preset generates ~200 customers and ~1,000 orders — fast and ideal for notebooks.

In [ ]:
from sqllocks_spindle import Spindle, RetailDomain

spindle = Spindle()
result = spindle.generate(domain=RetailDomain(), scale="fabric_demo", seed=42)
print(result)

In [ ]:
print(result.summary())

## Explore the tables

In [ ]:
result["customer"].head()

In [ ]:
result["order"].head()

## Address data — lat/lng ready for Power BI maps

In [ ]:
result["address"][["city", "state", "zip_code", "lat", "lng"]].head(10)

## Pareto order distribution

Spindle uses a Pareto distribution for customer orders — 20% of customers place ~80% of orders.

In [ ]:
order_counts = result["order"].groupby("customer_id").size().sort_values(ascending=False)
print(f"Max orders per customer:    {order_counts.max()}")
print(f"Median orders per customer: {order_counts.median():.1f}")
print(f"\nTop customers by order count:")
print(order_counts.head(10))

## Referential integrity

In [ ]:
errors = result.verify_integrity()
if errors:
    for e in errors:
        print(f"FK error: {e}")
else:
    print("Referential integrity: PASS — no FK violations")

## Distribution check — loyalty tiers

In [ ]:
result["customer"]["loyalty_tier"].value_counts(normalize=True).mul(100).round(1).rename("pct %")

## Override distributions at runtime

Pass `overrides={}` to simulate different business scenarios without editing any config files.

In [ ]:
# Simulate a premium-skewed customer base (e.g. loyalty program re-launch)
premium_domain = RetailDomain(overrides={
    "customer.loyalty_tier": {"Basic": 0.15, "Silver": 0.25, "Gold": 0.35, "Platinum": 0.25},
})

r2 = spindle.generate(domain=premium_domain, scale="fabric_demo", seed=42)
r2["customer"]["loyalty_tier"].value_counts(normalize=True).mul(100).round(1).rename("pct %")

In [ ]:
# Reproducibility — same seed always gives identical output
r3 = spindle.generate(domain=RetailDomain(), scale="fabric_demo", seed=42)
r4 = spindle.generate(domain=RetailDomain(), scale="fabric_demo", seed=42)
assert list(r3["customer"]["customer_id"]) == list(r4["customer"]["customer_id"])
print("Same seed → identical output: PASS")